# 68. 군집 전문가: 단기 흔들림을 장기 자기노이즈로 정규화
현재단일값군집이잘분리되지않아앞10샷흔들림/앞100샷흔들림,현재변화/앞100샷흔들림의8개특징을시험한다.100샷이력은현재샷제외·최소20샷·설비/제품/30분공백구간내에서만사용한다. 참조시각보다앞의비라벨관측을활용하는특징으로단위를자동수정하지않는다.
각외부평가구간정답은정규화학습·가스군집·모델·임계값선택에서제외한다. 이미전체자료를여러번봤으므로새로운미사용데이터독립검증아니다. 기존37개를개발자료로사용한다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s67=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
prefix=s67.split('F=pd.DataFrame')[0]
exec(prefix)
# Current and historical ratios use ONLY preceding observations within sensor-history segment.
ratios={}
keys=['EQUIP_CD','product','TimeStamp']
for sensor in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']:
 prior=observed.groupby('history_segment')[sensor].shift()
 scale=prior.groupby(observed.history_segment).transform(lambda a:a.rolling(100,min_periods=20).std())
 observed[sensor+'__past100_std_new']=scale
 fields=d[keys].merge(observed[keys+[sensor+'__past100_std_new']],on=keys,validate='one_to_one')
 den=fields[sensor+'__past100_std_new'].where(fields[sensor+'__past100_std_new']>1e-8)
 ratios[sensor+'__jitter10_over100']=d[sensor+'__past10_std']/den
 ratios[sensor+'__delta_over100']=d[sensor+'__delta1']/den
F=pd.DataFrame(ratios).replace([np.inf,-np.inf],np.nan)
F.assign(TimeStamp=d.TimeStamp).to_csv(tables/'rg3_history_ratio_features.csv',index=False,encoding='utf-8-sig')
tail=s67[s67.index("imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(F[y==0])"):]
tail=tail.replace('rg3_gas_positive_mode_diagnostic.csv','rg3_history_ratio_positive_mode_diagnostic.csv').replace('rg3_recall_first_mode_','rg3_history_ratio_mode_')
exec(tail)
